# Lahore AQI Predictor - Training Pipeline

**Project:** Pearls AQI Predictor | **Step:** Model Training

| Step | What happens |
|------|--------------|
| 1 | Install dependencies |
| 2 | Load features from Hopsworks or CSV |
| 3 | EDA - plots and stats |
| 4 | Feature selection and chronological split |
| 5 | Ridge Regression baseline |
| 6 | Random Forest |
| 7 | LSTM deep learning |
| 8 | Compare all models (RMSE, MAE, R2) |
| 9 | SHAP feature importance |
| 10 | Save best model to Hopsworks |
| 11 | AQI alert system |


## Step 1 - Install Dependencies

In [ ]:
!pip install -q hopsworks shap scikit-learn pandas numpy matplotlib seaborn tensorflow joblib
print('All packages installed')

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, json, logging, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timezone
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import shap

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (14, 5)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s')
logger = logging.getLogger(__name__)
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"Imports done | TF version: {tf.__version__}")


## Step 2A - Set API Keys

Paste your Hopsworks API key below, or set `USE_HOPSWORKS = False` and upload the CSV manually.


In [ ]:
USE_HOPSWORKS = True   # set False to upload CSV manually

HOPSWORKS_API_KEY     = "YOUR_HOPSWORKS_API_KEY_HERE"
FEATURE_GROUP_NAME    = "lahore_aqi_features"
FEATURE_GROUP_VERSION = 1


## Step 2B - Load Feature Data

In [ ]:
def load_from_hopsworks(api_key, fg_name, fg_version):
    import hopsworks
    logger.info("Connecting to Hopsworks...")
    project = hopsworks.login(api_key_value=api_key)
    fs = project.get_feature_store()
    fg = fs.get_feature_group(name=fg_name, version=fg_version)
    df = fg.read()
    logger.info("Loaded %d rows from '%s'", len(df), fg_name)
    return df

def load_from_csv(path):
    df = pd.read_csv(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    logger.info("Loaded %d rows from CSV", len(df))
    return df

if USE_HOPSWORKS:
    df_raw = load_from_hopsworks(HOPSWORKS_API_KEY, FEATURE_GROUP_NAME, FEATURE_GROUP_VERSION)
else:
    from google.colab import files
    print("Please upload lahore_features.csv ...")
    uploaded = files.upload()
    df_raw = load_from_csv(list(uploaded.keys())[0])

df_raw = df_raw.sort_values("timestamp").reset_index(drop=True)
print(f"Shape      : {df_raw.shape}")
print(f"Date range : {df_raw['timestamp'].min()} to {df_raw['timestamp'].max()}")
df_raw.head(3)


## Step 3 - Exploratory Data Analysis

In [ ]:
print("=" * 60)
print("  MISSING VALUES (%)")
print("=" * 60)
missing = (df_raw.isnull().mean() * 100).round(2)
problems = missing[missing > 0]
print(problems.to_string() if len(problems) > 0 else "  None")

print("\n" + "=" * 60)
print("  AQI TARGET STATS")
print("=" * 60)
for col in ["us_aqi", "target_aqi_24h", "target_aqi_48h", "target_aqi_72h"]:
    if col in df_raw.columns:
        s = df_raw[col].describe()
        print(f"  {col:<22}  mean={s['mean']:.1f}  std={s['std']:.1f}  max={s['max']:.0f}")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(df_raw["timestamp"], df_raw["us_aqi"], lw=0.6, color="steelblue")
axes[0].set_title("Lahore AQI - Full Historical Series", fontsize=13, fontweight='bold')
axes[0].set_ylabel("US AQI")
axes[0].axhline(150, color='orange', ls='--', lw=1, label='Unhealthy (150)')
axes[0].axhline(200, color='red',    ls='--', lw=1, label='Very Unhealthy (200)')
axes[0].legend(fontsize=9)

rolling_7d = df_raw.set_index("timestamp")["us_aqi"].rolling("7D").mean()
axes[1].plot(rolling_7d.index, rolling_7d.values, lw=1.2, color="darkorange", label="7-day rolling avg")
axes[1].set_title("7-Day Rolling Average AQI", fontsize=13, fontweight='bold')
axes[1].set_ylabel("US AQI")
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))

plt.tight_layout()
plt.savefig("aqi_timeseries.png", bbox_inches='tight')
plt.show()
print("Saved: aqi_timeseries.png")


In [ ]:
df_eda = df_raw.copy()
df_eda["month"] = df_eda["timestamp"].dt.month
df_eda["hour"]  = df_eda["timestamp"].dt.hour
df_eda["dow"]   = df_eda["timestamp"].dt.dayofweek

month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
dow_names   = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

monthly = df_eda.groupby("month")["us_aqi"].mean()
axes[0].bar(monthly.index, monthly.values, color="steelblue")
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(month_names, rotation=45)
axes[0].set_title("Avg AQI by Month", fontweight='bold')
axes[0].set_ylabel("Mean AQI")

hourly = df_eda.groupby("hour")["us_aqi"].mean()
axes[1].plot(hourly.index, hourly.values, marker='o', color="darkorange", lw=2)
axes[1].set_title("Avg AQI by Hour of Day", fontweight='bold')
axes[1].set_xlabel("Hour (UTC)")

weekly = df_eda.groupby("dow")["us_aqi"].mean()
axes[2].bar(weekly.index, weekly.values, color="mediumseagreen")
axes[2].set_xticks(range(7))
axes[2].set_xticklabels(dow_names)
axes[2].set_title("Avg AQI by Day of Week", fontweight='bold')

plt.suptitle("Lahore AQI - Seasonal and Diurnal Patterns", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("aqi_patterns.png", bbox_inches='tight')
plt.show()


In [ ]:
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
corr_cols = [c for c in numeric_cols
             if "target" not in c
             and c not in ["hour", "day_of_week", "month", "day_of_year", "is_weekend"]]

corr = df_raw[corr_cols].corr()["us_aqi"].drop("us_aqi").sort_values(ascending=False)
bar_colors = ["steelblue" if v > 0 else "tomato" for v in corr.values]

plt.figure(figsize=(10, 6))
corr.plot(kind='barh', color=bar_colors)
plt.axvline(0, color='black', lw=0.8)
plt.title("Feature Correlation with us_aqi", fontsize=13, fontweight='bold')
plt.xlabel("Pearson r")
plt.tight_layout()
plt.savefig("feature_correlation.png", bbox_inches='tight')
plt.show()

print("Top 5 positive:")
print(corr.head(5).to_string())
print("\nTop 5 negative:")
print(corr.tail(5).to_string())


## Step 4 - Feature Selection and Train/Val/Test Split

Never shuffle time-series data. Always split chronologically:
- **70%** train (oldest data)
- **15%** validation
- **15%** test (most recent data)


In [ ]:
TARGET_COL = "target_aqi_24h"   # change to target_aqi_48h or target_aqi_72h as needed

EXCLUDE = [
    "timestamp", "city",
    "target_aqi_24h", "target_aqi_48h", "target_aqi_72h",
    "us_aqi",
    "us_aqi_pm2_5",
]

FEATURE_COLS = [c for c in df_raw.columns if c not in EXCLUDE]
print(f"Feature columns ({len(FEATURE_COLS)}):")
print(FEATURE_COLS)

df_model = df_raw[FEATURE_COLS + [TARGET_COL, "timestamp"]].dropna(subset=[TARGET_COL])
df_model = df_model.dropna(subset=FEATURE_COLS)
print(f"\nRows available for modelling: {len(df_model):,}")


In [ ]:
n = len(df_model)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

df_train = df_model.iloc[:train_end]
df_val   = df_model.iloc[train_end:val_end]
df_test  = df_model.iloc[val_end:]

X_train = df_train[FEATURE_COLS].values
y_train = df_train[TARGET_COL].values
X_val   = df_val[FEATURE_COLS].values
y_val   = df_val[TARGET_COL].values
X_test  = df_test[FEATURE_COLS].values
y_test  = df_test[TARGET_COL].values

print(f"Train : {df_train['timestamp'].min().date()} to {df_train['timestamp'].max().date()}  ({len(df_train):>6,} rows)")
print(f"Val   : {df_val['timestamp'].min().date()} to {df_val['timestamp'].max().date()}   ({len(df_val):>6,} rows)")
print(f"Test  : {df_test['timestamp'].min().date()} to {df_test['timestamp'].max().date()}  ({len(df_test):>6,} rows)")


## Step 5 - Baseline: Ridge Regression

In [ ]:
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"  {name:<28}  RMSE={rmse:6.2f}  MAE={mae:6.2f}  R2={r2:.4f}")
    return {"model": name, "rmse": rmse, "mae": mae, "r2": r2}

results = []

ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge",  Ridge(alpha=1.0, random_state=SEED)),
])
ridge_pipe.fit(X_train, y_train)

print("=" * 65)
print("  RIDGE REGRESSION")
print("=" * 65)
results.append(evaluate("Ridge Val",  y_val,  ridge_pipe.predict(X_val)))
results.append(evaluate("Ridge Test", y_test, ridge_pipe.predict(X_test)))

joblib.dump(ridge_pipe, "ridge_model.pkl")
print("\nSaved: ridge_model.pkl")


## Step 6 - Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=4,
    n_jobs=-1,
    random_state=SEED,
)
rf_model.fit(X_train, y_train)

print("=" * 65)
print("  RANDOM FOREST")
print("=" * 65)
results.append(evaluate("RandomForest Val",  y_val,  rf_model.predict(X_val)))
results.append(evaluate("RandomForest Test", y_test, rf_model.predict(X_test)))

joblib.dump(rf_model, "random_forest_model.pkl")
print("\nSaved: random_forest_model.pkl")


In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
top20 = importances.sort_values(ascending=True).tail(20)

plt.figure(figsize=(10, 8))
top20.plot(kind='barh', color='steelblue')
plt.title("Random Forest - Top 20 Feature Importances", fontsize=13, fontweight='bold')
plt.xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.savefig("rf_importance.png", bbox_inches='tight')
plt.show()


## Step 7 - LSTM (Deep Learning)

LSTM treats feature rows as a sequence. It sees how AQI evolved over the past N hours,
which tabular models miss entirely.


In [ ]:
SEQ_LEN = 24   # look back 24 hours

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_s = scaler_X.fit_transform(X_train)
X_val_s   = scaler_X.transform(X_val)
X_test_s  = scaler_X.transform(X_test)

y_train_s = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_val_s   = scaler_y.transform(y_val.reshape(-1, 1)).ravel()

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i - seq_len : i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = make_sequences(X_train_s, y_train_s, SEQ_LEN)
X_val_seq,   y_val_seq   = make_sequences(X_val_s,   y_val_s,   SEQ_LEN)
X_test_seq,  y_test_seq  = make_sequences(X_test_s,  y_test,    SEQ_LEN)

print(f"LSTM input shape: {X_train_seq.shape}")
print(f"  (samples, seq_len={SEQ_LEN}, n_features={X_train_seq.shape[2]})")


In [ ]:
n_features = X_train_seq.shape[2]

lstm_model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN, n_features)),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(64, return_sequences=False),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),
], name="AQI_LSTM")

lstm_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

lstm_model.summary()


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=8, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4, verbose=1
    ),
]

history = lstm_model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=60,
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(history.history['loss'],     label='Train loss')
ax.plot(history.history['val_loss'], label='Val loss', ls='--')
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("LSTM Training Curve", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig("lstm_training_curve.png", bbox_inches='tight')
plt.show()


In [ ]:
lstm_pred_scaled = lstm_model.predict(X_test_seq, verbose=0).ravel()
lstm_pred = scaler_y.inverse_transform(lstm_pred_scaled.reshape(-1, 1)).ravel()

print("=" * 65)
print("  LSTM")
print("=" * 65)
results.append(evaluate("LSTM Test", y_test_seq, lstm_pred))

lstm_model.save("lstm_model.keras")
joblib.dump(scaler_X, "scaler_X.pkl")
joblib.dump(scaler_y, "scaler_y.pkl")
print("\nSaved: lstm_model.keras, scaler_X.pkl, scaler_y.pkl")


## Step 8 - Compare All Models

In [ ]:
results_df = pd.DataFrame(results)
print("\n" + "=" * 65)
print("  FINAL MODEL COMPARISON")
print("=" * 65)
print(results_df.to_string(index=False))

test_results = results_df[results_df["model"].str.contains("Test")]
best_row = test_results.loc[test_results["rmse"].idxmin()]
print(f"\nBest model: {best_row['model']}  (RMSE={best_row['rmse']:.2f}, R2={best_row['r2']:.4f})")


In [ ]:
rf_pred_test = rf_model.predict(X_test)
n_plot = min(168, len(y_test))
idx = range(n_plot)

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(idx, y_test[-n_plot:],       lw=1.5, label='Actual AQI',    color='steelblue')
axes[0].plot(idx, rf_pred_test[-n_plot:], lw=1.5, ls='--', label='RF Predicted', color='tomato')
axes[0].set_title("Random Forest - Predicted vs Actual (Last 7 Days of Test Set)", fontweight='bold')
axes[0].set_ylabel("US AQI")
axes[0].legend()

residuals = y_test[-n_plot:] - rf_pred_test[-n_plot:]
bar_colors = ["steelblue" if v > 0 else "tomato" for v in residuals]
axes[1].bar(idx, residuals, color=bar_colors, alpha=0.7)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title("Residuals (Actual minus Predicted)", fontweight='bold')
axes[1].set_ylabel("AQI Error")
axes[1].set_xlabel("Hour")

plt.tight_layout()
plt.savefig("rf_predictions.png", bbox_inches='tight')
plt.show()


## Step 9 - SHAP Feature Importance

In [ ]:
SHAP_SAMPLE = min(500, len(X_test))
X_shap = X_test[:SHAP_SAMPLE]

explainer   = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_shap)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=FEATURE_COLS, show=False, max_display=20)
plt.title("SHAP Summary - Random Forest (AQI 24h ahead)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("shap_summary.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: shap_summary.png")


In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, feature_names=FEATURE_COLS,
                  plot_type="bar", show=False, max_display=15)
plt.title("SHAP Feature Importance - Mean |SHAP value|", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("shap_bar.png", bbox_inches='tight')
plt.show()


## Step 10 - Save Best Model to Hopsworks Model Registry

In [ ]:
def save_to_hopsworks_registry(api_key, model_path, model_name, metrics,
                                description="", extra_files=None):
    try:
        import hopsworks
    except ImportError:
        logger.error("hopsworks not installed.")
        return

    import shutil
    logger.info("Connecting to Hopsworks Model Registry...")
    project = hopsworks.login(api_key_value=api_key)
    mr = project.get_model_registry()

    staging_dir = f"/tmp/{model_name}_staging"
    os.makedirs(staging_dir, exist_ok=True)
    shutil.copy(model_path, staging_dir)

    with open(f"{staging_dir}/feature_cols.json", "w") as fh:
        json.dump(FEATURE_COLS, fh)

    if extra_files:
        for fpath in extra_files:
            if os.path.exists(fpath):
                shutil.copy(fpath, staging_dir)

    model_obj = mr.python.create_model(
        name=model_name,
        metrics=metrics,
        description=description,
        input_example=X_test[:1].tolist(),
    )
    model_obj.save(staging_dir)
    logger.info("Saved '%s' to Hopsworks Model Registry v%s", model_name, model_obj.version)


if USE_HOPSWORKS:
    test_df = results_df[results_df["model"].str.contains("Test")]
    best    = test_df.loc[test_df["rmse"].idxmin()]
    print(f"Registering best model: {best['model']}")

    if "LSTM" in best["model"]:
        save_to_hopsworks_registry(
            api_key=HOPSWORKS_API_KEY,
            model_path="lstm_model.keras",
            model_name="lahore_aqi_lstm",
            metrics={"rmse": round(best["rmse"], 2), "mae": round(best["mae"], 2), "r2": round(best["r2"], 4)},
            description="LSTM 24h AQI prediction model for Lahore.",
            extra_files=["scaler_X.pkl", "scaler_y.pkl"],
        )
    else:
        save_to_hopsworks_registry(
            api_key=HOPSWORKS_API_KEY,
            model_path="random_forest_model.pkl",
            model_name="lahore_aqi_rf",
            metrics={"rmse": round(best["rmse"], 2), "mae": round(best["mae"], 2), "r2": round(best["r2"], 4)},
            description="Random Forest 24h AQI prediction model for Lahore.",
        )
else:
    print("Hopsworks skipped - models saved locally only.")


## Step 11 - AQI Alert System

In [ ]:
AQI_CATEGORIES = [
    (0,   50,  "GREEN   | Good"),
    (51,  100, "YELLOW  | Moderate"),
    (101, 150, "ORANGE  | Unhealthy for Sensitive Groups"),
    (151, 200, "RED     | Unhealthy"),
    (201, 300, "PURPLE  | Very Unhealthy"),
    (301, 500, "MAROON  | Hazardous"),
]

def classify_aqi(value):
    for lo, hi, label in AQI_CATEGORIES:
        if lo <= value <= hi:
            return label
    return "Unknown"

def generate_alert(city, aqi_24h, aqi_48h=None, aqi_72h=None):
    print(f"\n{'='*55}")
    print(f"  AQI FORECAST ALERT -- {city.upper()}")
    print(f"{'='*55}")
    print(f"  24h forecast: {aqi_24h:5.0f}  |  {classify_aqi(aqi_24h)}")
    if aqi_48h is not None:
        print(f"  48h forecast: {aqi_48h:5.0f}  |  {classify_aqi(aqi_48h)}")
    if aqi_72h is not None:
        print(f"  72h forecast: {aqi_72h:5.0f}  |  {classify_aqi(aqi_72h)}")
    if aqi_24h > 200:
        print("\n  HAZARD ALERT: Very Unhealthy / Hazardous levels forecast!")
        print("  - Avoid all outdoor activities")
        print("  - Keep windows closed, use air purifiers indoors")
        print("  - Wear N95 masks if you must go outside")
    elif aqi_24h > 150:
        print("\n  WARNING: Unhealthy AQI forecast.")
        print("  - Sensitive groups should avoid outdoor exertion")
        print("  - Consider working from home if possible")
    else:
        print("\n  Air quality forecast within acceptable range.")

latest_pred = float(rf_model.predict(X_test[-1:])[0])
generate_alert(city="Lahore", aqi_24h=latest_pred)


## Step 12 - Final Summary

In [ ]:
print("=" * 65)
print("  TRAINING PIPELINE - COMPLETE SUMMARY")
print("=" * 65)
print(f"  Target variable : {TARGET_COL}")
print(f"  Features used   : {len(FEATURE_COLS)}")
print(f"  Train rows      : {len(df_train):,}")
print(f"  Test rows       : {len(df_test):,}")
print()

print("  MODEL SCORES (Test Set):")
test_only = results_df[results_df['model'].str.contains('Test')].copy()
test_only['rmse'] = test_only['rmse'].round(2)
test_only['mae']  = test_only['mae'].round(2)
test_only['r2']   = test_only['r2'].round(4)
print(test_only.to_string(index=False))

print()
print("  SAVED FILES:")
for fname in ["ridge_model.pkl", "random_forest_model.pkl", "lstm_model.keras",
              "scaler_X.pkl", "scaler_y.pkl",
              "aqi_timeseries.png", "aqi_patterns.png",
              "rf_predictions.png", "shap_summary.png", "shap_bar.png"]:
    tag = "OK     " if os.path.exists(fname) else "MISSING"
    print(f"    [{tag}]  {fname}")

print()
print("  Training pipeline complete!")
print("  Next step: inference_pipeline.py then streamlit_app.py")


## What comes next?

| Step | File | What it does |
|------|------|--------------|
| Done | backfill_open_meteo.py | Historical data fetch |
| Done | feature_pipeline.py | Feature engineering + Hopsworks upload |
| Done | This notebook | Model training + registry |
| Next | inference_pipeline.py | Fetch live AQI, predict, store |
| Next | streamlit_app.py | Dashboard with live + forecast AQI |
| Next | .github/workflows/ | GitHub Actions: hourly feature, daily training |

### Multi-station average fix for live_aqi_client.py

Add this inside your fetch function to average all Lahore stations:

```python
stations = ["lahore", "lahore/us-consulate", "lahore/gulberg"]
readings = [fetch_aqi(s)["aqi"] for s in stations]
valid    = [r for r in readings if r is not None]
avg_aqi  = sum(valid) / len(valid) if valid else None
```
